# Model Inference

Loads the best model from MLflow Model Registry, predicts on the raw Walmart test set, and creates a Kaggle submission file.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from dataloader import load_raw, make_submission

DATA_DIR = ROOT / "data"
OUTPUT_PATH = ROOT / "models" / "best_model_submission.csv"

## MLflow / DagsHub Setup

Run this cell after the best model has been registered as `walmart-best` in the Model Registry.

In [ ]:
import mlflow

try:
    import dagshub
    dagshub.init(repo_owner="karev23", repo_name="ML-final", mlflow=True)
except ImportError:
    print("dagshub is not installed. If running locally, make sure MLFLOW_TRACKING_URI is configured.")

In [ ]:
train_raw, test_raw, features_df, stores_df, sample_submission = load_raw(DATA_DIR)

print("train:", train_raw.shape)
print("test:", test_raw.shape)
print("features:", features_df.shape)
print("stores:", stores_df.shape)

## Load Registered Best Model

Use `Production` if your registry uses stages. If DagsHub uses aliases in your setup, replace this with the champion alias URI.

In [ ]:
MODEL_URI = "models:/walmart-best/Production"

model = mlflow.pyfunc.load_model(MODEL_URI)
model

In [ ]:
preds = model.predict(test_raw)
preds = np.clip(np.asarray(preds, dtype=float), 0, None)

submission = make_submission(test_raw, preds, OUTPUT_PATH)
submission.head()

In [ ]:
print(f"Saved submission to: {OUTPUT_PATH}")
print(submission.shape)